In [ ]:
import sys
sys.path.insert(0, '../main')
sys.path.insert(0, '../main/utils')


In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

import config

2026-05-27 12:44:40.723940: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-27 12:44:40.740220: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-27 12:44:40.740263: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-27 12:44:40.755900: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-27 12:44:46.041828: W tensorflow/compiler/tf


[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0



2026-05-27 12:44:57.161923: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-27 12:44:57.219450: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-27 12:44:57.220709: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [2]:

exp_folder = config.DEFAULT_EXP_FOLDER
exp_path = Path(exp_folder, 'D20250808_E00_C00_F4500KHz_U_Sample_7')
train_data_path = os.path.join(exp_path, config.TRAINING_DATA_PATH)

print(f"[*] Loading data from: {train_data_path}")
pipeline_state = joblib.load(train_data_path)

timestamps = pipeline_state["timestamps"]
ori_idx = list(pipeline_state["dataset_name"]).index("ori_curves")
ori_curves = pipeline_state["dataset"][ori_idx]
features_df = pipeline_state["kinetic_features"][ori_idx]
Y_well = pipeline_state["Y_well"]

# Spatial metadata
row_idx = pipeline_state["metadata_df"]["pixel_row_idx"].values
col_idx = pipeline_state["metadata_df"]["pixel_col_idx"].values

max_row = 49
max_col = 81
T = len(timestamps)

unique_wells = np.unique(Y_well)
num_wells = len(unique_wells)
well_to_idx = {w: i for i, w in enumerate(unique_wells)}

[*] Loading data from: /vol/bitbucket/gk225/POC_DDM_dataset/D20250808_E00_C00_F4500KHz_U_Sample_7/curve_for_training.joblib


In [3]:
print(f"[*] Reconstructing spatial grids for {num_wells} wells...")

# Scale curves to [0, 1] for stable Autoencoder training
min_val = np.min(ori_curves)
max_val = np.max(ori_curves)
scaled_curves = (ori_curves - min_val) / (max_val - min_val + 1e-9)

# Initialize blank video tensor: (Samples, Time, Height, Width, Channels)
X_grid = np.zeros((num_wells, T, max_row, max_col, 1), dtype=np.float32)

# Scatter the 1D curves into their exact hardware coordinates
for i in range(len(scaled_curves)):
    w_idx = well_to_idx[Y_well[i]]
    r = row_idx[i]
    c = col_idx[i]
    X_grid[w_idx, :, r, c, 0] = scaled_curves[i]

print(f"  -> Grid shape constructed: {X_grid.shape}")

[*] Reconstructing spatial grids for 10 wells...
  -> Grid shape constructed: (10, 615, 49, 81, 1)


In [4]:
print("[*] Building ConvLSTM Autoencoder...")

def build_convlstm_ae(time_steps, rows, cols):
    model = models.Sequential([
        layers.Input(shape=(time_steps, rows, cols, 1)),
        
        # Encoder
        layers.ConvLSTM2D(filters=16, kernel_size=(3, 3), padding='same', return_sequences=True),
        layers.BatchNormalization(),
        
        # Bottleneck
        layers.ConvLSTM2D(filters=8, kernel_size=(3, 3), padding='same', return_sequences=True),
        layers.BatchNormalization(),
        
        # Decoder
        layers.ConvLSTM2D(filters=16, kernel_size=(3, 3), padding='same', return_sequences=True),
        layers.BatchNormalization(),
        
        # Output Layer
        layers.ConvLSTM2D(filters=1, kernel_size=(3, 3), padding='same', return_sequences=True, activation='linear')
    ])
    
    model.compile(optimizer='adam', loss='mse')
    return model

tf.keras.backend.clear_session()
autoencoder = build_convlstm_ae(T, max_row, max_col)

[*] Building ConvLSTM Autoencoder...


2026-05-27 12:45:12.148954: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-27 12:45:12.150316: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-27 12:45:12.151527: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [5]:
# ====================================================================
# 4. TRAIN AND INFER
# ====================================================================
print("[*] Training ConvLSTM Autoencoder...")
early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

autoencoder.fit(
    X_grid, X_grid,
    epochs=100,
    batch_size=2,
    callbacks=[early_stop],
    verbose=1
)

print("[*] Predicting and extracting per-pixel anomalies...")
X_pred = autoencoder.predict(X_grid, batch_size=2)

[*] Training ConvLSTM Autoencoder...
Epoch 1/100


2026-05-27 12:45:16.978197: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'Func/StatefulPartitionedCall/gradient_tape/sequential_1/conv_lstm2d_2_1/while/sequential_1/conv_lstm2d_2_1/while_grad/body/_761/input/_1686' -> 'StatefulPartitionedCall/gradient_tape/sequential_1/conv_lstm2d_2_1/while/sequential_1/conv_lstm2d_2_1/while_grad/body/_761/gradient_tape/sequential_1/conv_lstm2d_2_1/while/gradients/AddN', 'Func/StatefulPartitionedCall/gradient_tape/sequential_1/conv_lstm2d_1_2/while/sequential_1/conv_lstm2d_1_2/while_grad/body/_911/input/_1778' -> 'StatefulPartitionedCall/gradient_tape/sequential_1/conv_lstm2d_1_2/while/sequential_1/conv_lstm2d_1_2/while_grad/body/_911/gradient_tape/sequential_1/conv_lstm2d_1_2/while/gradients/AddN', 'Func/StatefulPartitionedCall/gradient_tape/sequential_1/conv_lstm2d_1/while/sequential_1/conv_lstm2d_1/while_grad/body/_1061/inp

5/5 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - loss: nan
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step - loss: nan
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - loss: nan
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step - loss: nan
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step - loss: nan
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - loss: nan
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - loss: nan
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step - loss: nan
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step - loss: nan
Epoch 10/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step - loss: nan
Epoch 11/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - loss: nan
[*] Predicting and extracting per-pixel anomalies...
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 770ms/step


In [6]:
# ====================================================================
# 5. ERROR EXTRACTION & MULTIPLE THRESHOLDING
# ====================================================================
mse_errors = np.zeros(len(ori_curves))

# Extract spatial errors specifically for the active pixels
for i in range(len(ori_curves)):
    w_idx = well_to_idx[Y_well[i]]
    r = row_idx[i]
    c = col_idx[i]
    
    true_curve = X_grid[w_idx, :, r, c, 0]
    pred_curve = X_pred[w_idx, :, r, c, 0]
    mse_errors[i] = np.mean((true_curve - pred_curve) ** 2)

# Method A: 95th Percentile
threshold_95 = np.percentile(mse_errors, 95)
outlier_labels_95 = (mse_errors > threshold_95).astype(int)

# Method B: Geometric Elbow Method
def calculate_elbow_threshold(errors):
    """
    Calculates the point of maximum curvature (the elbow) on the sorted error curve
    using perpendicular geometric distance.
    """
    sorted_errors = np.sort(errors)
    n_points = len(sorted_errors)
    
    # Create coordinates for the curve points
    points = np.column_stack((np.arange(n_points), sorted_errors))
    
    # Line vector from first point to last point
    p1 = points[0]
    p2 = points[-1]
    line_vec = p2 - p1
    line_vec_norm = line_vec / np.linalg.norm(line_vec)
    
    # Vector from p1 to all points
    vec_from_first = points - p1
    
    # Calculate scalar and vector projections
    scalar_proj = np.dot(vec_from_first, line_vec_norm)
    vec_proj = np.outer(scalar_proj, line_vec_norm)
    
    # The elbow is the point furthest from the straight line (maximum perpendicular distance)
    perp_dist = np.linalg.norm(vec_from_first - vec_proj, axis=1)
    elbow_idx = np.argmax(perp_dist)
    
    return sorted_errors[elbow_idx]

elbow_threshold = calculate_elbow_threshold(mse_errors)
outlier_labels_elbow = (mse_errors > elbow_threshold).astype(int)

print(f"  -> Threshold (95%): {threshold_95:.7f} | Flagged {np.sum(outlier_labels_95)} outliers")
print(f"  -> Threshold (Elbow): {elbow_threshold:.7f} | Flagged {np.sum(outlier_labels_elbow)} outliers")

  -> Threshold (95%): nan | Flagged 0 outliers
  -> Threshold (Elbow): nan | Flagged 0 outliers


In [7]:
print("[*] Saving results back to pipeline state...")

# Append new columns to the features dataframe
features_df["convlstm_ae_mse"] = mse_errors
features_df["convlstm_ae_label_95"] = outlier_labels_95
features_df["convlstm_ae_label_elbow"] = outlier_labels_elbow

# Overwrite the dataframe in the pipeline state
pipeline_state["kinetic_features"][ori_idx] = features_df

# Save back to disk
joblib.dump(pipeline_state, train_data_path, compress=3)

print(f"[✓] Successfully updated {config.TRAINING_DATA_PATH}!")

[*] Saving results back to pipeline state...
[✓] Successfully updated curve_for_training.joblib!
